# Lab 5


Matrix Representation: In this lab you will be creating a simple linear algebra system. In memory, we will represent matrices as nested python lists as we have done in lecture. In the exercises below, you are required to explicitly test every feature you implement, demonstrating it works.

1. Create a `matrix` class with the following properties:
    * It can be initialized in 2 ways:
        1. with arguments `n` and `m`, the size of the matrix. A newly instanciated matrix will contain all zeros.
        2. with a list of lists of values. Note that since we are using lists of lists to implement matrices, it is possible that not all rows have the same number of columns. Test explicitly that the matrix is properly specified.
    * Matrix instances `M` can be indexed with `M[i][j]` and `M[i,j]`.
    * (Matrix assignment works in 2 ways:
        1. If `M_1` and `M_2` are `matrix` instances `M_1=M_2` sets the values of `M_1` to those of `M_2`, if they are the same size. Error otherwise.
        2. In example above `M_2` can be a list of lists of correct size.)


In [178]:
class Matrix:
    def __init__(self, n, m=None):
        """Initializing a matrix with (n, m) size or a list of lists."""
        if isinstance(n, list):  
            # If given a list of lists, using it as matrix data
            self.data = n  
            self.rows = len(n)
            self.cols = len(n[0]) if self.rows > 0 else 0  

            # Checking if all rows have the same number of columns
            for row in n:
                if len(row) != self.cols:
                    raise ValueError("Rows must have the same number of columns")
                    
        elif isinstance(n, int) and isinstance(m, int):
            # Creating an n x m zero matrix
            self.rows = n
            self.cols = m
            self.data = [[0] * m for _ in range(n)]
        else:
            raise ValueError("Invalid input. Must be (rows, cols) or a list of lists.")

    def __getitem__(self, index):
        """Allows M[i][j] and M[i, j] indexing."""
        if isinstance(index, tuple):
            return self.data[index[0]][index[1]]
        return self.data[index]

    def __setitem__(self, index, value):
        """Allows assignment using both M[i][j] and M[i, j]."""
        if isinstance(index, tuple):
            self.data[index[0]][index[1]] = value
        else:
            self.data[index] = value

    def __repr__(self):
        """Returns the matrix in a readable format."""
        return "\n".join(str(row) for row in self.data)


# testing

print("Creating a 3x3 zero matrix:")
M1 = Matrix(3, 3)
print(M1)

print("\nCreating a matrix from a list of lists:")
M2 = Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print(M2)

print("\nTesting indexing:")
print("M2[1][2]:", M2[1][2])  
print("M2[1,2]:", M2[1,2])  

# Invalid matrix test (should ideally raise an error)
try:
    print("\nTrying to create an invalid matrix...")
    M_invalid = Matrix([[1, 2], [3, 4, 5]])  # Here Rows have different lengths hence it should raise an error
except ValueError as e:
    print("Error caught as expected:", e)

Creating a 3x3 zero matrix:
[0, 0, 0]
[0, 0, 0]
[0, 0, 0]

Creating a matrix from a list of lists:
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]

Testing indexing:
M2[1][2]: 6
M2[1,2]: 6

Trying to create an invalid matrix...
Error caught as expected: Rows must have the same number of columns


2. Add the following methods:
    * `shape()`: returns a tuple `(n,m)` of the shape of the matrix.
    * `transpose()`: returns a new matrix instance which is the transpose of the matrix.
    * `row(n)` and `column(n)`: that return the nth row or column of the matrix M as a new appropriately shaped matrix object.
    * `to_list()`: which returns the matrix as a list of lists.
    *  `block(n_0,n_1,m_0,m_1)` that returns a smaller matrix located at the n_0 to n_1 columns and m_0 to m_1 rows. 
    * (Extra credit) Modify `__getitem__` implemented above to support slicing.
        

In [180]:
class Matrix:
    def __init__(self, n, m=None):
        """Initializing a matrix either with (n, m) size or a list of lists."""
        if isinstance(n, list):  
            # If given a list of lists, using it as the matrix
            self.data = n  
            self.rows = len(n)
            self.cols = len(n[0]) if self.rows > 0 else 0  

            # Making sure all rows have the same number of columns
            for row in n:
                if len(row) != self.cols:
                    raise ValueError("Rows must have the same number of columns")
                    
        elif isinstance(n, int) and isinstance(m, int):
            # Creaating an n x m zero matrix
            self.rows = n
            self.cols = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]
        else:
            raise ValueError("Invalid input. Must be (rows, cols) or a list of lists.")

    def __getitem__(self, index):
        """Allows indexing using M[i][j], M[i, j], and slicing for rows."""
        if isinstance(index, tuple):
            return self.data[index[0]][index[1]]
        elif isinstance(index, slice):  # Extra credit work shown here 
            return Matrix(self.data[index])
        return self.data[index]

    def __setitem__(self, index, value):
        """Allows assignment using both M[i][j] and M[i, j]."""
        if isinstance(index, tuple):
            self.data[index[0]][index[1]] = value
        else:
            self.data[index] = value

    def __repr__(self):
        """Returns the matrix in a readable format."""
        return "\n".join(str(row) for row in self.data)

    def shape(self):
        """Returns the number of rows and columns (rows, cols)."""
        return self.rows, self.cols

    def transpose(self):
        """Returns a new matrix that is the transpose (flip rows and columns)."""
        return Matrix([[self.data[j][i] for j in range(self.rows)] for i in range(self.cols)])

    def row(self, n):
        """Returns the nth row as a new matrix."""
        if n < 0 or n >= self.rows:
            raise IndexError("Row index out of range")
        return Matrix([self.data[n]])

    def column(self, n):
        """Returns the nth column as a new matrix."""
        if n < 0 or n >= self.cols:
            raise IndexError("Column index out of range")
        return Matrix([[self.data[i][n]] for i in range(self.rows)])

    def to_list(self):
        """Returns the matrix as a list of lists."""
        return [row[:] for row in self.data]

    def block(self, n_0, n_1, m_0, m_1):
        """Returns a submatrix from row indices (n_0 to n_1) and column indices (m_0 to m_1)."""
        if not (0 <= n_0 < n_1 <= self.rows and 0 <= m_0 < m_1 <= self.cols):
            raise ValueError("Invalid block range")
        return Matrix([row[m_0:m_1] for row in self.data[n_0:n_1]])


# testing
M = Matrix([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
print("Original Matrix:")
print(M)

print("\nShape of the matrix:")
print(M.shape())

print("\nTranspose of the matrix:")
print(M.transpose())

print("\nRow 1:")
print(M.row(1))

print("\nColumn 2:")
print(M.column(2))

print("\nMatrix converted to a list of lists:")
print(M.to_list())

print("\nExtracting a Block (rows 1 to 3, columns 0 to 2):")
print(M.block(1, 3, 0, 2))

# Testing the slicing feature (extra credit work shown here again)
print("\nMatrix Slice (first two rows):")
print(M[:2])

Original Matrix:
[1, 2, 3]
[4, 5, 6]
[7, 8, 9]

Shape of the matrix:
(3, 3)

Transpose of the matrix:
[1, 4, 7]
[2, 5, 8]
[3, 6, 9]

Row 1:
[4, 5, 6]

Column 2:
[3]
[6]
[9]

Matrix converted to a list of lists:
[[1, 2, 3], [4, 5, 6], [7, 8, 9]]

Extracting a Block (rows 1 to 3, columns 0 to 2):
[4, 5]
[7, 8]

Matrix Slice (first two rows):
[1, 2, 3]
[4, 5, 6]


3. Write functions that create special matrices (note these are standalone functions, not member functions of your `matrix` class):
    * `constant(n,m,c)`: returns a `n` by `m` matrix filled with floats of value `c`.
    * `zeros(n,m)` and `ones(n,m)`: return `n` by `m` matrices filled with floats of value `0` and `1`, respectively.
    * `eye(n)`: returns the n by n identity matrix.

In [182]:
def constant(n, m, c):
    """Createing an n x m matrix filled with a constant value c (as a float)."""
    return [[float(c) for _ in range(m)] for _ in range(n)]

def zeros(n, m):
    """Creating an n x m matrix filled with 0s."""
    return constant(n, m, 0.0)  

def ones(n, m):
    """Creating an n x m matrix filled with 1s."""
    return constant(n, m, 1.0)  

def eye(n):
    """Creating an n x n identity matrix (1s on the diagonal, 0s elsewhere)."""
    I = [[0.0 for _ in range(n)] for _ in range(n)]
    for i in range(n):
        I[i][i] = 1.0  
    return I

# testing
const_matrix = constant(3, 4, 2.5)
zero_matrix = zeros(3, 4)
one_matrix = ones(3, 4)
identity_matrix = eye(4)

# results
print("Constant Matrix (3x4, value 2.5):")
for row in const_matrix:
    print(row)

print("\nZero Matrix (3x4):")
for row in zero_matrix:
    print(row)

print("\nOne Matrix (3x4):")
for row in one_matrix:
    print(row)

print("\nIdentity Matrix (4x4):")
for row in identity_matrix:
    print(row)

Constant Matrix (3x4, value 2.5):
[2.5, 2.5, 2.5, 2.5]
[2.5, 2.5, 2.5, 2.5]
[2.5, 2.5, 2.5, 2.5]

Zero Matrix (3x4):
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0]

One Matrix (3x4):
[1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.0, 1.0]
[1.0, 1.0, 1.0, 1.0]

Identity Matrix (4x4):
[1.0, 0.0, 0.0, 0.0]
[0.0, 1.0, 0.0, 0.0]
[0.0, 0.0, 1.0, 0.0]
[0.0, 0.0, 0.0, 1.0]


4. Add the following member functions to your class. Make sure to appropriately test the dimensions of the matrices to make sure the operations are correct.
    * `M.scalarmul(c)`: a matrix that is scalar product $cM$, where every element of $M$ is multiplied by $c$.
    * `M.add(N)`: adds two matrices $M$ and $N$. Don’t forget to test that the sizes of the matrices are compatible for this and all other operations.
    * `M.sub(N)`: subtracts two matrices $M$ and $N$.
    * `M.mat_mult(N)`: returns a matrix that is the matrix product of two matrices $M$ and $N$.
    * `M.element_mult(N)`: returns a matrix that is the element-wise product of two matrices $M$ and $N$.
    * `M.equals(N)`: returns true/false if $M==N$.

In [184]:
class Matrix:
    def __init__(self, n, m=None):
        """Initializing a matrix with (n, m) size or a list of lists."""
        if isinstance(n, list):  
            # Using list of lists as the matrix
            self.data = n  
            self.rows = len(n)
            self.cols = len(n[0]) if self.rows > 0 else 0  

            # Checking if all rows have the same number of columns
            for row in n:
                if len(row) != self.cols:
                    raise ValueError("All rows must have the same number of columns")
                    
        elif isinstance(n, int) and isinstance(m, int):
            # Creating an n x m zero matrix
            self.rows = n
            self.cols = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]
        else:
            raise ValueError("Invalid input. Must be (rows, cols) or a list of lists.")

    
    def __repr__(self):
        """Prints the matrix in a readable format."""
        return "\n".join(str(row) for row in self.data)

    def scalarmul(self, c):
        """Multiplies every element in the matrix by scalar c."""
        return Matrix([[self.data[i][j] * c for j in range(self.cols)] for i in range(self.rows)])

    def add(self, N):
        """Adds two matrices (element-wise)."""
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match for addition.")
        return Matrix([[self.data[i][j] + N.data[i][j] for j in range(self.cols)] for i in range(self.rows)])

    def sub(self, N):
        """Subtracts one matrix from another (element-wise)."""
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match for subtraction.")
        return Matrix([[self.data[i][j] - N.data[i][j] for j in range(self.cols)] for i in range(self.rows)])

    def mat_mult(self, N):
        """Performs matrix multiplication."""
        if self.cols != N.rows:
            raise ValueError("Number of columns in first matrix must match rows in second matrix.")
        
        result = [[0 for _ in range(N.cols)] for _ in range(self.rows)]
        for i in range(self.rows):
            for j in range(N.cols):
                result[i][j] = sum(self.data[i][k] * N.data[k][j] for k in range(self.cols))
        
        return Matrix(result)

    
    def element_mult(self, N):
        """Performing element-wise multiplication."""
        if self.shape() != N.shape():
            raise ValueError("Matrix dimensions must match for element-wise multiplication.")
        return Matrix([[self.data[i][j] * N.data[i][j] for j in range(self.cols)] for i in range(self.rows)])

    def equals(self, N):
        """Checking if two matrices are the same."""
        return self.data == N.data

    def shape(self):
        """Returns (rows, cols) of the matrix."""
        return self.rows, self.cols



# testing
M1 = Matrix([[1, 2], [3, 4]])
M2 = Matrix([[5, 6], [7, 8]])

print("Matrix M1:")
print(M1)

print("\nScalar Multiplication of M1 by 2:")
print(M1.scalarmul(2))

print("\nAddition (M1 + M2):")
print(M1.add(M2))

print("\nSubtraction (M1 - M2):")
print(M1.sub(M2))

print("\nMatrix Multiplication (M1 * M2):")
print(M1.mat_mult(M2))

print("\nElement-wise Multiplication (M1 * M2):")
print(M1.element_mult(M2))

print("\nChecking if M1 is equal to M2:")
print(M1.equals(M2))

print("\nChecking if M1 is equal to itself:")
print(M1.equals(M1))

Matrix M1:
[1, 2]
[3, 4]

Scalar Multiplication of M1 by 2:
[2, 4]
[6, 8]

Addition (M1 + M2):
[6, 8]
[10, 12]

Subtraction (M1 - M2):
[-4, -4]
[-4, -4]

Matrix Multiplication (M1 * M2):
[19, 22]
[43, 50]

Element-wise Multiplication (M1 * M2):
[5, 12]
[21, 32]

Checking if M1 is equal to M2:
False

Checking if M1 is equal to itself:
True


5. Overload python operators to appropriately use your functions in 4 and allow expressions like:
    * 2*M
    * M*2
    * M+N
    * M-N
    * M*N
    * M==N
    * M=N


In [186]:
class Matrix:
    def __init__(self, n, m=None):
        """Initializing a matrix with (n, m) size or a list of lists."""
        if isinstance(n, list):  
            # Using list of lists as the matrix
            self.data = n  
            self.rows = len(n)
            self.cols = len(n[0]) if self.rows > 0 else 0  

            # Making sure all rows have the same number of columns
            for row in n:
                if len(row) != self.cols:
                    raise ValueError("Rows must have the same number of columns")
                    
        elif isinstance(n, int) and isinstance(m, int):
            # Creating an n x m zero matrix
            self.rows = n
            self.cols = m
            self.data = [[0 for _ in range(m)] for _ in range(n)]
        else:
            raise ValueError("Invalid input. Must be (rows, cols) or a list of lists.")

    def __repr__(self):
        """Prints the matrix in a readable format."""
        return "\n".join(str(row) for row in self.data)

    def shape(self):
        """Returns the dimensions of the matrix (rows, cols)."""
        return self.rows, self.cols


    
    # Overloaded Operators

    def __mul__(self, other):
        """Handling scalar multiplication (M * 2) and matrix multiplication (M * N)."""
        if isinstance(other, (int, float)):  # Scalar multiplication
            return Matrix([[self.data[i][j] * other for j in range(self.cols)] for i in range(self.rows)])
        elif isinstance(other, Matrix):  # Matrix multiplication
            if self.cols != other.rows:
                raise ValueError("Number of columns in first matrix must match rows in second matrix.")
            
            result = [[0 for _ in range(other.cols)] for _ in range(self.rows)]
            for i in range(self.rows):
                for j in range(other.cols):
                    result[i][j] = sum(self.data[i][k] * other.data[k][j] for k in range(self.cols))
            
            return Matrix(result)
        else:
            raise TypeError("Multiplication not supported for this type.")

    def __rmul__(self, other):
        """Handling scalar multiplication from the left (2 * M)."""
        return self.__mul__(other)

    def __add__(self, other):
        """Handling matrix addition (M + N)."""
        if not isinstance(other, Matrix) or self.shape() != other.shape():
            raise ValueError("Matrices must have the same dimensions for addition.")
        return Matrix([[self.data[i][j] + other.data[i][j] for j in range(self.cols)] for i in range(self.rows)])

    def __sub__(self, other):
        """Handling matrix subtraction (M - N)."""
        if not isinstance(other, Matrix) or self.shape() != other.shape():
            raise ValueError("Matrices must have the same dimensions for subtraction.")
        return Matrix([[self.data[i][j] - other.data[i][j] for j in range(self.cols)] for i in range(self.rows)])

    def __eq__(self, other):
        """Checking if two matrices are equal (M == N)."""
        return isinstance(other, Matrix) and self.data == other.data

    def __copy__(self):
        """Creating a copy of the matrix (M = N)."""
        return Matrix([row[:] for row in self.data])




# testing
M1 = Matrix([[1, 2], [3, 4]])
M2 = Matrix([[5, 6], [7, 8]])

print("Matrix M1:")
print(M1)

print("\nScalar Multiplication (M1 * 2):")
print(M1 * 2)

print("\nScalar Multiplication (2 * M1):")
print(2 * M1)

print("\nAddition (M1 + M2):")
print(M1 + M2)

print("\nSubtraction (M1 - M2):")
print(M1 - M2)

print("\nMatrix Multiplication (M1 * M2):")
print(M1 * M2)

print("\nChecking if M1 is equal to M2:")
print(M1 == M2)

print("\nChecking if M1 is equal to itself:")
print(M1 == M1)

# assignment test (M = N)
M3 = M1.__copy__()
print("\nAssignment Test (M3 = M1):")
print(M3)

Matrix M1:
[1, 2]
[3, 4]

Scalar Multiplication (M1 * 2):
[2, 4]
[6, 8]

Scalar Multiplication (2 * M1):
[2, 4]
[6, 8]

Addition (M1 + M2):
[6, 8]
[10, 12]

Subtraction (M1 - M2):
[-4, -4]
[-4, -4]

Matrix Multiplication (M1 * M2):
[19, 22]
[43, 50]

Checking if M1 is equal to M2:
False

Checking if M1 is equal to itself:
True

Assignment Test (M3 = M1):
[1, 2]
[3, 4]


6. Demonstrate the basic properties of matrices with your matrix class by creating two 2 by 2 example matrices using your Matrix class and illustrating the following:

$$
(AB)C=A(BC)
$$
$$
A(B+C)=AB+AC
$$
$$
AB\neq BA
$$
$$
AI=A
$$

In [188]:
# demonstrating basic matrix properties using the Matrix class

# Defining some example of 2x2 matrices
A = Matrix([[1, 2], [3, 4]])
B = Matrix([[5, 6], [7, 8]])
C = Matrix([[2, 0], [1, 3]])
I = Matrix([[1, 0], [0, 1]])  # Identity matrix shown here


# Property 1: (AB)C = A(BC) (Associative Property)
left_side = (A * B) * C
right_side = A * (B * C)
associative_property = left_side == right_side  


# Property 2: A(B + C) = AB + AC (Distributive Property)
left_side = A * (B + C)
right_side = (A * B) + (A * C)
distributive_property = left_side == right_side  


# Property 3: AB ≠ BA (Matrix multiplication is NOT commutative)
commutative_property = (A * B) == (B * A)  


# Property 4: AI = A (Multiplicative Identity Property)
identity_property = (A * I) == A  


# results
print("Checking basic matrix properties:\n")

print("1) Associativity: (AB)C == A(BC) →", associative_property)
print("2) Distributivity: A(B + C) == AB + AC →", distributive_property)
print("3) Commutativity: AB == BA →", commutative_property, "(Expected: False)")
print("4) Identity Property: AI == A →", identity_property)

Checking basic matrix properties:

1) Associativity: (AB)C == A(BC) → True
2) Distributivity: A(B + C) == AB + AC → True
3) Commutativity: AB == BA → False (Expected: False)
4) Identity Property: AI == A → True
